# CS 260C — Qwen2.5-1.5B compression eval (Colab)

End-to-end runner for `run_qwen_compression.py`. Runs the **Wanda** pruning and **AWQ** INT4 quantization experiments on `Qwen/Qwen2.5-1.5B-Instruct`, evaluates each on GSM8K and HumanEval, and saves JSON result files alongside the existing baseline runs in `results/`.

**Before running:** set `Runtime > Change runtime type > GPU` (T4 is fine; A100/L4 is faster). AWQ needs CUDA so this notebook will not work on a CPU runtime.

## 0. Sanity-check the GPU

In [ ]:
!nvidia-smi

## 1. Clone (or update) the project repo

Edit `REPO_URL` if you're working off a fork. The clone is idempotent — re-running the cell just `git pull`s.

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/kriteenjain/COMSCI260c-project.git"
REPO_DIR = "/content/COMSCI260c-project"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!ls

## 2. Install dependencies

- `requirements.txt`: torch / transformers / datasets / accelerate / tqdm (most are pre-installed in Colab; this just upgrades).
- `autoawq`: only needed for the AWQ cell. Pinned `>=0.2.6` for Qwen2.5 support.
- `tabulate`: optional, for the summary table at the bottom.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q "autoawq>=0.2.6"
!pip install -q tabulate

## 3. (Optional) Hugging Face login

Only required if you want to swap `--model` for a gated model like `meta-llama/Llama-2-7b-hf`. Qwen2.5 is public and works without a token.

Uncomment the cell and paste a *read* token when prompted.

In [ ]:
# from huggingface_hub import login
# login()

## 4. Shared knobs

Bump `LIMIT` for paper-quality numbers (full GSM8K = 1319, full HumanEval = 164, pass `-1`). Start small for a smoke test.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TASK = "both"           # 'gsm8k' | 'humaneval' | 'both'
LIMIT = 50              # -1 for the full split
BATCH_SIZE = 4          # bump to 8 on A100 / L4
COMPRESSED_DIR = "/content/compressed"  # where pruned / quantized models are cached

import os
os.makedirs(COMPRESSED_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)
print({"MODEL": MODEL, "TASK": TASK, "LIMIT": LIMIT, "BATCH_SIZE": BATCH_SIZE})

## 5. (Optional) Re-run the FP16 baseline

Skip if you already have a baseline JSON in `results/` from a previous Colab session — every compressed run produces a result file with the same schema, so you only need one baseline at the same `LIMIT` to diff against.

In [ ]:
!python run_baseline.py \
    --model {MODEL} \
    --task {TASK} \
    --limit {LIMIT} \
    --batch-size {BATCH_SIZE} \
    --tag colab-baseline{LIMIT}

## 6. Wanda — unstructured 50% sparsity

Calibrates on 128 WikiText-2 windows of 2048 tokens and prunes every `nn.Linear` in the decoder blocks per-output-row by `|W| * ||X||_2`. ~3–5 min on a T4 for Qwen2.5-1.5B.

`--save-compressed` caches the pruned weights so cell 8 can re-eval without re-pruning.

In [ ]:
WANDA_U50_DIR = f"{COMPRESSED_DIR}/qwen-wanda-u50"

!python run_qwen_compression.py \
    --model {MODEL} \
    --method wanda \
    --sparsity-type unstructured \
    --sparsity-ratio 0.5 \
    --task {TASK} \
    --limit {LIMIT} \
    --batch-size {BATCH_SIZE} \
    --save-compressed {WANDA_U50_DIR}

## 7. Wanda — 2:4 structured

Same algorithm, but every 4 consecutive weights in a row must have at least 2 zeros — this is the pattern modern NVIDIA GPUs can actually accelerate (`SparseTensor`). Useful for the speedup story in the writeup.

In [ ]:
WANDA_24_DIR = f"{COMPRESSED_DIR}/qwen-wanda-2-4"

!python run_qwen_compression.py \
    --model {MODEL} \
    --method wanda \
    --sparsity-type 2:4 \
    --task {TASK} \
    --limit {LIMIT} \
    --batch-size {BATCH_SIZE} \
    --save-compressed {WANDA_24_DIR}

## 8. AWQ — INT4, group size 128

First run: AWQ search + pseudo-quantize + pack INT4 weights, then evaluate. Takes ~5–10 min on a T4. The packed weights are cached in `{COMPRESSED_DIR}/qwen-awq-w4g128` so future re-evals skip the search.

In [ ]:
AWQ_DIR = f"{COMPRESSED_DIR}/qwen-awq-w4g128"

!python run_qwen_compression.py \
    --model {MODEL} \
    --method awq \
    --w-bit 4 \
    --q-group-size 128 \
    --task {TASK} \
    --limit {LIMIT} \
    --batch-size {BATCH_SIZE} \
    --save-compressed {AWQ_DIR}

## 9. (Optional) Re-evaluate a cached compressed model

Once `--save-compressed` has written a directory, you can skip the compression step entirely. Useful when you want to bump `--limit` for the final numbers without paying the compression cost again.

In [ ]:
# Example: full HumanEval on the cached AWQ model.
# !python run_qwen_compression.py \
#     --method awq --task humaneval --limit -1 \
#     --load-compressed {AWQ_DIR} --tag awq-w4g128-full

## 10. Summary table

Reads every JSON in `results/` and tabulates GSM8K accuracy and HumanEval pass@1 side-by-side. Sorted newest-first.

In [ ]:
import json, glob, os
from tabulate import tabulate

rows = []
for path in sorted(glob.glob("results/*.json"), reverse=True):
    with open(path) as f:
        r = json.load(f)
    tasks = r.get("tasks", {})
    gsm = tasks.get("gsm8k", {})
    he = tasks.get("humaneval", {})
    rows.append([
        os.path.basename(path),
        r.get("tag", ""),
        r.get("method", "baseline"),
        r.get("limit", ""),
        f"{gsm.get('accuracy', float('nan')):.3f}" if gsm else "-",
        f"{he.get('pass@1', float('nan')):.3f}" if he else "-",
        round(r.get("elapsed_s", 0), 1),
    ])

print(tabulate(
    rows,
    headers=["file", "tag", "method", "limit", "gsm8k acc", "humaneval pass@1", "elapsed (s)"],
    tablefmt="github",
))

## 11. (Optional) Persist results to Google Drive

Colab wipes `/content` between sessions. Mount Drive and copy `results/` (and optionally `{COMPRESSED_DIR}`) over so you don't lose the JSON files — these are what `scripts/compare.py` will read for the final writeup.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/cs260c_results
# !cp -r results/* /content/drive/MyDrive/cs260c_results/
# !ls /content/drive/MyDrive/cs260c_results

## 12. (Optional) Commit results back to the repo

Quick way to share runs with the rest of the group. Configure your name/email + a personal access token first. **Be careful** — this pushes from the Colab VM.

In [ ]:
# !git config user.email "you@example.com"
# !git config user.name  "Your Name"
# !git add results/
# !git commit -m "colab: add compressed Qwen runs"
# !git push